<a href="https://colab.research.google.com/github/Joaoplims/NLP-HandsOn/blob/main/HOF02/HOF02_IntentionalityFilter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
def convert_github_blob_to_raw(url: str) -> str:
    """
    Converts a GitHub blob URL to a raw usercontent URL.
    Example:
    From: https://github.com/user/repo/blob/main/file.csv
    To:   https://raw.githubusercontent.com/user/repo/main/file.csv
    """
    if "github.com" in url and "/blob/" in url:
        url = url.replace("github.com", "raw.githubusercontent.com")
        url = url.replace("/blob/", "/")
    return url

# Example usage:
test_url = "https://github.com/Joaoplims/NLP-HandsOn/blob/main/HOF02/spam.csv"
raw_url = convert_github_blob_to_raw(test_url)
print(f"Raw URL: {raw_url}")

Raw URL: https://raw.githubusercontent.com/Joaoplims/NLP-HandsOn/main/HOF02/spam.csv


In [51]:
import pandas as pd
from datasets import load_dataset
import kagglehub
import os
import glob

def process_url_and_create_df(dataset_item: dict) -> pd.DataFrame | None:
    """
    Loads data based on the source specified in the dataset_item dictionary.
    Supports: 'Github', 'Hugging Face', and 'Kaggle'.
    """
    source = dataset_item.get('source')
    address = dataset_item.get('address')

    try:
        if source == "Github":
            print(f"Downloading directly from GitHub: {address}")
            try:
                return pd.read_csv(address)
            except UnicodeDecodeError:
                print("UTF-8 decoding failed. Retrying with latin-1 encoding...")
                return pd.read_csv(address, encoding='latin-1')

        elif source == "Hugging Face":
            print(f"Loading Hugging Face dataset: {address}")
            ds = load_dataset(address)
            if hasattr(ds, 'keys'):
                split_name = 'train' if 'train' in ds else list(ds.keys())[0]
                return ds[split_name].to_pandas()
            return ds.to_pandas()

        elif source == "Kaggle":
            print(f"Downloading Kaggle dataset: {address}")
            path = kagglehub.dataset_download(address)
            print(f"Path to dataset files: {path}")

            # Search for CSV or JSONL files
            csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
            jsonl_files = glob.glob(os.path.join(path, "**", "*.jsonl"), recursive=True)

            if csv_files:
                print(f"Found CSV: {os.path.basename(csv_files[0])}")
                return pd.read_csv(csv_files[0])
            elif jsonl_files:
                print(f"Found JSONL: {os.path.basename(jsonl_files[0])}")
                return pd.read_json(jsonl_files[0], lines=True)
            else:
                all_files = glob.glob(os.path.join(path, "**", "*"), recursive=True)
                print(f"No supported files found. Files available: {[os.path.basename(f) for f in all_files if os.path.isfile(f)]}")
                return None

        else:
            print(f"Unsupported source: {source}")
            return None

    except Exception as e:
        print(f"Error loading from {source} ({address}): {e}")
        return None

In [53]:
import pandas as pd

# Updated list with structured sources
dataset_urls = [
    {"source": "Github", "address": "https://raw.githubusercontent.com/Joaoplims/NLP-HandsOn/main/HOF02/spam.csv"},
    {"source": "Github", "address": "https://media.githubusercontent.com/media/RockENZO/data/refs/heads/main/Synthetic-Data-for-Scam-Detection-Leveraging-LLMs-to-Train-Deep-Learning-Models-main/data/single-agent-scam-dialogue_train.csv"},
    {"source": "Github", "address": "https://media.githubusercontent.com/media/RockENZO/data/refs/heads/main/Synthetic-Data-for-Scam-Detection-Leveraging-LLMs-to-Train-Deep-Learning-Models-main/data/multi_agent_conversation_train.csv"},
    {"source": "Hugging Face", "address": "wangyuancheng/discord-phishing-scam-clean"},
    {"source": "Kaggle", "address": "kanishkaranaweera/synthetic-dialogue-dataset-romance-scam-detection"}
]

indexDataset = 2
item = dataset_urls[indexDataset]
scam_conversations_df = process_url_and_create_df(item)

if scam_conversations_df is not None:
    print(f"Successfully loaded dataset from {item['source']}:")
    display(scam_conversations_df.head())
else:
    print(f"Failed to load the dataset from {item['address']}.")

Successfully loaded dataset from Github:


,dialogue,personality,type,labels
0,"Innocent: Hello. Suspect: Hi, this is Karen f...",aggressive,appointment,0
1,"Innocent: Hello. Suspect: Hi, this is Karen f...",aggressive,appointment,0
2,"Innocent: Hello. Suspect: Hi, this is Karen f...",aggressive,appointment,0
3,"Innocent: Hello. Suspect: Hi, this is Rachel ...",aggressive,appointment,0
4,"Innocent: Hello. Suspect: Hi, this is Karen f...",aggressive,appointment,0
